# Stop Times Checks

Checks trip duplication, stop_sequence regularity, and route-following behavior in stop_times.

In [1]:
import csv
import os
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import sys
from typing import Dict, List, Set, Tuple

_current = Path.cwd().resolve()
for _candidate in [_current, *_current.parents]:
    if (_candidate / "data_validation" / "checks" / "commons.py").exists():
        _project_root = _candidate
        break
else:
    raise FileNotFoundError("data_validation/checks/commons.py not found.")

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from data_validation.checks.commons import (
    BASE,
    STOP_TIMES_CLEANED_FILE,
    STOP_TIMES_FILE,
    STOPS_FILE,
    TRIPS_CLEANED_FILE,
    TRIPS_FILE,
    build_expected_adjacency,
    check_missing_files,
    check_trip,
    collect_trip_stop_ids,
    basics,
    make_signature,
    load_stop_names,
    load_trip_ids,
    sniff_dialect,
    read_dict_rows,
)

#### Duplicate full trip_id in stop_times

In [11]:
def main():
    check_missing_files([STOP_TIMES_FILE, TRIPS_FILE])

    # trip_id of interest (filtered by prefix '1.')
    trip_ids = load_trip_ids(TRIPS_FILE)
    trip_ids = sorted(ids for ids in trip_ids if ids.startswith("1."))
    trip_id_set = set(trip_ids)

    print(f"Unique trip_id in trips (prefix '1.'): {len(trip_ids)}")

    # Aggregate all stop_times rows per trip_id
    # Store tuples (stop_sequence_int, arrival_time, departure_time, stop_id)
    trips_rows: Dict[str, List[Tuple[int, str, str, str]]] = defaultdict(list)
    total_rows = 0
    for r in read_dict_rows(STOP_TIMES_FILE):
        total_rows += 1
        tid = r.get('trip_id', '')
        if tid not in trip_id_set:
            continue
        seq_s = r.get('stop_sequence', '')
        arr = r.get('arrival_time', '')
        dep = r.get('departure_time', '')
        sid = r.get('stop_id', '')
        try:
            seq = int(seq_s)
        except Exception:
            seq = 10**9
        trips_rows[tid].append((seq, arr, dep, sid))

    print(f"Total rows read from stop_times: {total_rows}")
    print(f"trip_id with at least one row in stop_times: {len(trips_rows)}")

    # Build signatures in parallel for each trip_id
    sig_to_trips: Dict[Tuple[Tuple[int, str, str, str], ...], List[str]] = defaultdict(list)
    max_workers = min(8, (os.cpu_count() or 4))
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(make_signature, item): item[0] for item in trips_rows.items()}
        for fut in as_completed(futures):
            tid, sig = fut.result()
            sig_to_trips[sig].append(tid)

    # Duplicate groups: signatures shared by >= 2 different trip_id
    duplicate_groups = [(sig, sorted(tids)) for sig, tids in sig_to_trips.items() if len(tids) >= 2]
    duplicate_groups.sort(key=lambda x: (len(x[1]), x[1]))

    if not duplicate_groups:
        print("No pair/group of trip_id with identical sequence and schedules was found.")
        return

    total_dup_trips = sum(len(tids) for _, tids in duplicate_groups)
    print(f"\nFound {len(duplicate_groups)} groups of trip_id with identical content ({total_dup_trips} trip_id in total).")

    len_counts = {}
    for _, tids in duplicate_groups:
        count = len(tids)
        len_counts[count] = len_counts.get(count, 0) + 1
    for count in sorted(len_counts.keys()):
        print(f"Groups with {count} trip_id: {len_counts[count]}")
        for sig, tids in duplicate_groups:
            if len(tids) == count:
                print(f"  Example group with {count} trip_id: {tids[:5]}{'...' if len(tids) > 5 else ''}")
                break

    # ---- Generate file with trip_id to eliminate ----
    # For each group, keep the first trip_id (alphabetically) and eliminate the rest
    to_eliminate: List[str] = []
    for _, tids in duplicate_groups:
        to_eliminate.extend(tids[1:])  # tids is already sorted

    out_path = os.path.join(BASE, "trip_ids_to_eliminate.txt")
    with open(out_path, "w", encoding="utf-8") as f:
        for tid in sorted(to_eliminate):
            f.write(tid + "\n")

    kept = total_dup_trips - len(to_eliminate)
    print(f"\nTrip_id kept from groups (1 per group): {kept}")
    print(f"Trip_id to eliminate: {len(to_eliminate)}")
    print(f"Total valid trip_id after removing duplicates: {len(trip_ids) - len(to_eliminate)}")
    print(f"Generated file: {out_path}")

    # ---- Clean stop_times using the elimination list (sorted by trip_id, stop_sequence) ----
    with open(out_path, 'r', encoding='utf-8') as f:
        eliminate_trip_ids: Set[str] = {line.strip() for line in f if line.strip()}

    stop_times_cleaned_path = os.path.join(BASE, "stop_times_cleaned.txt")
    dialect = sniff_dialect(STOP_TIMES_FILE)

    total_stop_times_rows = 0
    kept_stop_times_rows = 0
    removed_stop_times_rows = 0

    rows_to_keep: List[Dict[str, str]] = []
    with open(STOP_TIMES_FILE, 'r', encoding='utf-8-sig', newline='') as fin:
        reader = csv.DictReader(fin, dialect=dialect)
        if reader.fieldnames is None:
            raise RuntimeError("stop_times.txt has no header.")

        fieldnames = reader.fieldnames
        for row in reader:
            total_stop_times_rows += 1
            tid = row.get('trip_id', '').strip()
            if tid in eliminate_trip_ids:
                removed_stop_times_rows += 1
            else:
                rows_to_keep.append(row)
                kept_stop_times_rows += 1

    rows_to_keep.sort(
        key=lambda r: (
            r.get('trip_id', '') or '',
            int(r.get('stop_sequence', '') or 10**9) if str(r.get('stop_sequence', '') or '').isdigit() else 10**9,
        )
    )

    with open(stop_times_cleaned_path, 'w', encoding='utf-8', newline='') as fout:
        writer = csv.DictWriter(fout, fieldnames=fieldnames, dialect=dialect)
        writer.writeheader()
        for row in rows_to_keep:
            writer.writerow(row)

    print(f"Total rows in stop_times.txt: {total_stop_times_rows}")
    print(f"Rows removed: {removed_stop_times_rows}")
    print(f"Rows kept: {kept_stop_times_rows}")
    print(f"Cleaned file saved to: {stop_times_cleaned_path}")

    # ---- Clean trips using the same elimination list (sorted by route_id, trip_id, direction_id) ----
    trips_cleaned_path = os.path.join(BASE, "trips_cleaned.txt")
    trips_dialect = sniff_dialect(TRIPS_FILE)
    total_trips_rows = 0
    removed_trips_rows = 0

    # Collect rows to keep, then sort
    rows_to_keep: List[Dict[str, str]] = []
    with open(TRIPS_FILE, 'r', encoding='utf-8-sig', newline='') as fin:
        reader = csv.DictReader(fin, dialect=trips_dialect)
        if reader.fieldnames is None:
            raise RuntimeError("trips.txt has no header.")
        fieldnames = reader.fieldnames

        for row in reader:
            total_trips_rows += 1
            tid = row.get('trip_id', '').strip()
            if tid in eliminate_trip_ids:
                removed_trips_rows += 1
            else:
                rows_to_keep.append(row)

    # Sort by route_id, trip_id, direction_id
    rows_to_keep.sort(
        key=lambda r: (
            r.get('route_id', '') or '',
            r.get('trip_id', '') or '',
            r.get('direction_id', '') or ''
        )
    )
    kept_trips_rows = len(rows_to_keep)

    # Write sorted rows
    with open(trips_cleaned_path, 'w', encoding='utf-8', newline='') as fout:
        writer = csv.DictWriter(fout, fieldnames=fieldnames, dialect=trips_dialect)
        writer.writeheader()
        for row in rows_to_keep:
            writer.writerow(row)

    print(f"Total rows in trips.txt: {total_trips_rows}")
    print(f"Rows removed: {removed_trips_rows}")
    print(f"Rows kept: {kept_trips_rows}")
    print(f"Cleaned file saved to: {trips_cleaned_path}")

main()

Unique trip_id in trips (prefix '1.'): 15088
Total rows read from stop_times: 1138586
trip_id with at least one row in stop_times: 15088

Found 4085 groups of trip_id with identical content (8196 trip_id in total).
Groups with 2 trip_id: 4065
  Example group with 2 trip_id: ['1.1.11654310', '1.1.11655306']
Groups with 3 trip_id: 14
  Example group with 3 trip_id: ['1.101.11427287', '1.101.11427796', '1.101.11428435']
Groups with 4 trip_id: 6
  Example group with 4 trip_id: ['1.104.11428897', '1.104.11430235', '1.104.11571861', '1.104.11572496']

Trip_id kept from groups (1 per group): 4085
Trip_id to eliminate: 4111
Total valid trip_id after removing duplicates: 10977
Generated file: /Users/saradalmauguamis/Desktop/Mates/Cursos/Curs 2025-2026 (3r + 4t)/TFG/TFG/.src/gtfs/data/trip_ids_to_eliminate.txt
Total rows in stop_times.txt: 1138586
Rows removed: 78215
Rows kept: 1060371
Cleaned file saved to: /Users/saradalmauguamis/Desktop/Mates/Cursos/Curs 2025-2026 (3r + 4t)/TFG/TFG/.src/gtfs/

Validate stop_times_cleaned.txt and trips_cleaned.txt using trip_ids_to_eliminate.txt.

In [ ]:
def main():
    eliminate_path = os.path.join(BASE, "trip_ids_to_eliminate.txt")
    cleaned_stop_times_path = os.path.join(BASE, "stop_times_cleaned.txt")
    cleaned_trips_path = os.path.join(BASE, "trips_cleaned.txt")
    check_missing_files([eliminate_path, cleaned_stop_times_path, cleaned_trips_path])

    # Load trip_id values to eliminate once.
    with open(eliminate_path, 'r', encoding='utf-8') as f:
        to_eliminate: Set[str] = {line.strip() for line in f if line.strip()}

    print(f"Trip_id to eliminate: {len(to_eliminate)}")

    # 1) Validate that stop_times_cleaned does not contain eliminated trip_id.
    found_in_stop_times: Set[str] = set()
    total_stop_times_rows = 0
    for r in read_dict_rows(cleaned_stop_times_path):
        total_stop_times_rows += 1
        tid = r.get('trip_id', '').strip()
        if tid in to_eliminate:
            found_in_stop_times.add(tid)

    print(f"Total rows in stop_times_cleaned.txt: {total_stop_times_rows}")
    if not found_in_stop_times:
        print("All correct: no eliminated trip_id found in stop_times_cleaned.txt.")
    else:
        print(f"ERROR: {len(found_in_stop_times)} eliminated trip_id still present in stop_times_cleaned.txt:")
        for tid in sorted(found_in_stop_times):
            print(f"- {tid}")

    # 2) Validate that trips_cleaned does not contain eliminated trip_id.
    found_in_trips: Set[str] = set()
    total_trips_rows = 0
    for r in read_dict_rows(cleaned_trips_path):
        total_trips_rows += 1
        tid = r.get('trip_id', '').strip()
        if tid in to_eliminate:
            found_in_trips.add(tid)

    print(f"Total rows in trips_cleaned.txt: {total_trips_rows}")
    if not found_in_trips:
        print("All correct: no eliminated trip_id found in trips_cleaned.txt.")
    else:
        print(f"ERROR: {len(found_in_trips)} eliminated trip_id still present in trips_cleaned.txt:")
        for tid in sorted(found_in_trips):
            print(f"- {tid}")

main()

Trip_id to eliminate: 4111
Total rows in stop_times_cleaned.txt: 1060371
All correct: no eliminated trip_id found in stop_times_cleaned.txt.
Total rows in trips_cleaned.txt: 44876
All correct: no eliminated trip_id found in trips_cleaned.txt.


#### Does stop_sequence increment by one?

Goal: check whether stop_sequence in stop_times.txt increments by one. To do this, we look at all 
trip_id from trips.txt. For each one, we go to stop_times.txt and look at the rows with that 
trip_id. We aggregate only the stop_sequence values per trip_id (without storing the full row) and check 
whether they increment by one. Optimized: we read stop_times once and aggregate by trip_id.
Parallel validation per trip with ThreadPoolExecutor to reduce total time.

In [9]:
def main():
    check_missing_files([STOP_TIMES_CLEANED_FILE, TRIPS_CLEANED_FILE])

    # trip_id of interest (filtered by prefix '1.')
    trip_ids = load_trip_ids(TRIPS_CLEANED_FILE)
    trip_ids = sorted(ids for ids in trip_ids if ids.startswith("1."))
    trip_id_set = set(trip_ids)

    print(f"Unique trip_id in trips_cleaned (prefix '1.'): {len(trip_ids)}")

    # Build: trip_id -> set of stop_sequence (no duplicates), in a single pass over stop_times_cleaned
    seq_by_trip: Dict[str, Set[int]] = defaultdict(set)
    for r in read_dict_rows(STOP_TIMES_CLEANED_FILE):
        tid = r.get('trip_id', '')
        if tid not in trip_id_set:
            continue
        seq_str = r.get('stop_sequence', '')
        try:
            seq = int(seq_str)
        except Exception:
            # Ignore non-numeric or empty values
            continue
        seq_by_trip[tid].add(seq)

    # Final map: trip_id -> sorted list of stop_sequence (no duplicates)
    dict_seq: Dict[str, List[int]] = {}
    for trip_id in trip_ids:
        dict_seq[trip_id] = sorted(seq_by_trip.get(trip_id, set()))

    # Parallel validation
    max_workers = min(8, (os.cpu_count() or 4))
    violations_total = 0
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {
            ex.submit(check_trip, trip_id, dict_seq[trip_id]): trip_id for trip_id in trip_ids
        }
        for fut in as_completed(futures):
            msgs = fut.result()
            violations_total += len(msgs)
            for m in msgs:
                print(m)

    if violations_total == 0:
        print("Correct: all trips in trips_cleaned have stop_sequence that increments by one.")
    else:
        print(f"Total violations detected: {violations_total}")

main()

Unique trip_id in trips_cleaned (prefix '1.'): 10977
Correct: all trips in trips_cleaned have stop_sequence that increments by one.


#### The stop sequency is followed correctly?

Goal: detect whether there is a subway trip that skips a stop or changes the expected order.

Mechanism: We use dictionaries from scripts/basics.py. Each route_name has a route_id in 'subway_routes_names_ids'. For each route_id, we gather trip_id values (and direction_id) from 'trips_cleaned.txt'. For each trip_id, we read rows from 'stop_times_cleaned.txt' ordered by stop_sequence and compare each consecutive pair with the expected order from 'subway_route_names_stop_ids' (reversed when direction_id=1). If a pair is flagged, we print only the two stops in bad order with their stop_sequence values and stop names from 'stops.txt'.

In [2]:
def main():
    check_missing_files([TRIPS_CLEANED_FILE, STOP_TIMES_CLEANED_FILE, STOPS_FILE])

    subway_routes_names_ids = basics.subway_routes_names_ids
    subway_route_names_stop_ids = basics.subway_route_names_stop_ids

    # Build the exact route_id -> route_name mapping.
    rid_to_name = {rid: name for name, rid in subway_routes_names_ids.items()}

    def stop_sort_key(stop_id: str) -> Tuple[int, str]:
        _, _, suffix = stop_id.partition('.')
        try:
            return (int(suffix), stop_id)
        except Exception:
            return (10**9, stop_id)

    # Collect trip -> (route_id, direction_id) only for subway route_ids.
    trip_to_route_dir = {}
    matched_rows = 0
    for r in read_dict_rows(TRIPS_CLEANED_FILE):
        tid = r.get('trip_id', '').strip()
        if not tid:
            continue
        rid = r.get('route_id', '').strip()
        if rid not in rid_to_name:
            continue
        matched_rows += 1
        dir_id = r.get('direction_id', '').strip() or ''
        trip_to_route_dir[tid] = (rid, dir_id)

    if not trip_to_route_dir:
        available_route_ids = sorted({r.get('route_id', '').strip() for r in read_dict_rows(TRIPS_CLEANED_FILE) if r.get('route_id', '').strip()})
        suffix = '' if len(available_route_ids) <= 20 else ' ...'
        print('No subway trips found in trips_cleaned.txt for the canonical mappings.')
        print(f'Trips rows matching subway route_ids: {matched_rows}')
        print(f'Available route_id values in trips_cleaned.txt: {available_route_ids[:20]}{suffix}')
        return

    trip_ids = set(trip_to_route_dir.keys())

    # Aggregate ordered (stop_sequence, stop_id) lists per trip in one pass over stop_times_cleaned.
    trip_seq_rows: Dict[str, List[Tuple[int, str]]] = defaultdict(list)
    for r in read_dict_rows(STOP_TIMES_CLEANED_FILE):
        tid = r.get('trip_id', '').strip()
        if tid not in trip_ids:
            continue
        sid = r.get('stop_id', '').strip()
        if not sid:
            continue
        seq_s = r.get('stop_sequence', '').strip()
        try:
            seq = int(seq_s)
        except Exception:
            continue
        trip_seq_rows[tid].append((seq, sid))

    for tid in trip_seq_rows:
        trip_seq_rows[tid].sort(key=lambda x: x[0])

    stop_names = load_stop_names(STOPS_FILE)

    # (route_name, route_id, trip_id, direction_id, bad_pairs)
    # where each bad_pair is (seq_a, stop_a, seq_b, stop_b).
    flagged = []

    for tid, seq_rows in trip_seq_rows.items():
        rid, dir_id = trip_to_route_dir.get(tid, (None, None))
        if not rid:
            continue
        route_name = rid_to_name.get(rid)
        expected_set = subway_route_names_stop_ids.get(route_name, set())
        if not expected_set:
            continue
        expected = sorted(expected_set, key=stop_sort_key)
        expected_to_check = list(reversed(expected)) if dir_id == '1' else expected
        adj = build_expected_adjacency(expected_to_check)

        bad_pairs = []
        for i in range(max(0, len(seq_rows) - 1)):
            seq_a, a = seq_rows[i]
            seq_b, b = seq_rows[i + 1]
            if adj.get(a) != b:
                bad_pairs.append((seq_a, a, seq_b, b))

        if bad_pairs:
            flagged.append((route_name, rid, tid, dir_id, bad_pairs))

    if not flagged:
        print('All checked subway trips follow an allowed contiguous stop sequence.')
        return

    print(f'FOUND {len(flagged)} trips with unexpected stop sequences:')
    for route_name, rid, tid, dir_id, bad_pairs in flagged:
        print(f'- route={route_name!r} route_id={rid} trip_id={tid} direction={dir_id} bad_pairs={len(bad_pairs)}')
        for seq_a, a, seq_b, b in bad_pairs:
            name_a = stop_names.get(a, '(no name)')
            name_b = stop_names.get(b, '(no name)')
            print(
                f'    bad order: [{seq_a}] {a} ({name_a}) -> [{seq_b}] {b} ({name_b})'
            )

main()

FOUND 2510 trips with unexpected stop sequences:
- route='L10S' route_id=1.101.1 trip_id=1.101.11426484 direction=0 bad_pairs=1
    bad order: [5] 1.959 (Provençana) -> [6] 1.914 (Can Tries | Gornal)
- route='L10S' route_id=1.101.1 trip_id=1.101.11426485 direction=1 bad_pairs=1
    bad order: [3] 1.914 (Can Tries | Gornal) -> [4] 1.959 (Provençana)
- route='L10S' route_id=1.101.1 trip_id=1.101.11426486 direction=0 bad_pairs=1
    bad order: [5] 1.959 (Provençana) -> [6] 1.914 (Can Tries | Gornal)
- route='L10S' route_id=1.101.1 trip_id=1.101.11426487 direction=1 bad_pairs=1
    bad order: [3] 1.914 (Can Tries | Gornal) -> [4] 1.959 (Provençana)
- route='L10S' route_id=1.101.1 trip_id=1.101.11426488 direction=0 bad_pairs=1
    bad order: [5] 1.959 (Provençana) -> [6] 1.914 (Can Tries | Gornal)
- route='L10S' route_id=1.101.1 trip_id=1.101.11426489 direction=1 bad_pairs=1
    bad order: [3] 1.914 (Can Tries | Gornal) -> [4] 1.959 (Provençana)
- route='L10S' route_id=1.101.1 trip_id=1.101